# UrduStack — Train Risk Scorer (LoRA XLM-RoBERTa)

Run this notebook in Google Colab free-tier GPU (T4).

**What it does**
- Downloads the **Roman-Urdu-Toxic-Corpus** (72.7k rows, CC-BY-4.0) from Hugging Face automatically.
- Fine-tunes `xlm-roberta-base` on a 15k sample using LoRA.
- Saves a small LoRA adapter to `models/risk_lora/`.
- Computes a temperature-scaling value and writes it to `models/temperature.txt`.

**Expected total runtime on free Colab T4**
- Setup + installs: ~3–5 min
- Model download: ~2–5 min
- Training (3 epochs, 15k samples, batch 16): ~15–30 min
- Temperature calibration + test eval: ~1–2 min
- **Total: ~25–45 minutes**

**You do not need to download any dataset manually.** Cell 5 fetches it directly from Hugging Face.

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies (pinned versions to avoid PEFT / Python 3.13 crashes).
# Colab already has torch installed; we only add what the training script needs.
#
# IMPORTANT: drop torchao first. Colab ships torchao 0.10, but PEFT >=0.14
# aborts on import if it finds anything below 0.16. We don't need torchao
# for LoRA fine-tuning, so uninstalling it is the cleanest fix.
!pip uninstall -y torchao 2>/dev/null
!pip install -q -U \
  "transformers>=4.46.0" \
  "datasets>=3.1.0" \
  "peft>=0.13.2" \
  "accelerate>=1.1.0" \
  scikit-learn pandas

# Sanity-check the versions that matter
import peft, transformers, datasets
print(f"peft={peft.__version__}  transformers={transformers.__version__}  datasets={datasets.__version__}")

In [ ]:
# Optional: mount Google Drive so the trained adapter is saved permanently
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the UrduStack repo (idempotent — handles re-runs and nested clones).
import os, shutil

# If a previous run accidentally cloned UrduStack inside UrduStack, remove it.
if os.path.exists('UrduStack/UrduStack'):
    shutil.rmtree('UrduStack/UrduStack')
    print('Removed nested UrduStack/UrduStack')

if os.path.exists('UrduStack'):
    %cd UrduStack
    !git pull
else:
    !git clone https://github.com/munazat/UrduStack.git
    %cd UrduStack

!pwd   # should print /content/UrduStack

In [ ]:
# Download the Roman-Urdu-Toxic-Corpus from Hugging Face and save as CSV.
# Source: https://huggingface.co/datasets/hafiz-hassaan-saeed/Roman-Urdu-Toxic-Corpus
# 72,771 rows | columns: Roman_Urdu, Toxic | license: CC-BY-4.0
import os
from datasets import load_dataset

os.makedirs('data/raw', exist_ok=True)

if not os.path.exists('data/raw/PURUTT.csv'):
    print('Downloading Roman-Urdu-Toxic-Corpus from Hugging Face...')
    ds = load_dataset('hafiz-hassaan-saeed/Roman-Urdu-Toxic-Corpus', split='train')
    df = ds.to_pandas()

    # Rename columns to match what the training script expects
    df = df.rename(columns={'Roman_Urdu': 'text', 'Toxic': 'label'})

    df.to_csv('data/raw/PURUTT.csv', index=False)
    print(f'Downloaded {len(df)} rows. Saved to data/raw/PURUTT.csv')
else:
    print('PURUTT.csv already present — skipping download.')

In [ ]:
# Inspect the dataset
import pandas as pd
df = pd.read_csv('data/raw/PURUTT.csv')
print(df.head())
print(df.columns.tolist())
print(df['label'].value_counts())

In [ ]:
# Train
!python scripts/train_risk_model.py \
  --data_path data/raw/PURUTT.csv \
  --output_dir models/risk_lora \
  --max_samples 15000 \
  --val_samples 2000 \
  --test_samples 2000 \
  --num_epochs 3 \
  --batch_size 16

In [ ]:
# Check outputs
import os
print('Adapter files:', os.listdir('models/risk_lora'))
if os.path.exists('models/temperature.txt'):
    print('Temperature:', open('models/temperature.txt').read().strip())

In [ ]:
# Optional: push adapter to Hugging Face Hub
# !huggingface-cli login
# !python scripts/train_risk_model.py \
#   --data_path data/raw/PURUTT.csv \
#   --output_dir models/risk_lora \
#   --push_to_hub \
#   --hub_model_id your-username/urdustack-risk-lora

In [ ]:
# Optional: copy results to Drive (only if Drive was mounted)
import os, shutil

drive_dest = '/content/drive/MyDrive/urdustack_models'
if os.path.exists('/content/drive/MyDrive'):
    shutil.copytree('models', drive_dest, dirs_exist_ok=True)
    print('Copied models to', drive_dest)
else:
    print('Drive not mounted — skipping copy. Use the Files panel to download models/ manually.')

In [ ]:
# Launch live demo with a public URL (share=True gives you a *.gradio.live link)
# The model loads on Colab's T4 GPU — inference is fast.
# This URL works as long as the Colab notebook is running (~72 hours).
import sys
sys.path.insert(0, '/content/UrduStack')

!pip install -q gradio

from playground import analyze
import gradio as gr

demo = gr.Interface(
    fn=analyze,
    inputs=gr.Textbox(
        label="Input (Urdu / Roman Urdu / English mix)",
        placeholder="yar bhai I'm bohat pareshan aaj...",
        lines=3,
    ),
    outputs=[
        gr.Textbox(label="Normalized Urdu", interactive=False),
        gr.Textbox(label="Risk Score", interactive=False),
        gr.Markdown(label="Analysis"),
    ],
    title="UrduStack — Code-Switch-Aware Urdu NLP",
    description=(
        "Type mixed-script Urdu text and see the normalized output "
        "plus an explainable risk/toxicity score powered by a fine-tuned XLM-RoBERTa LoRA adapter."
    ),
    examples=[
        ["yar mujhe pareshan mat karo bro"],
        ["job available, 50000 per week, send processing fee"],
        ["aaj weather bohat achha hai"],
        ["bhai ye to scam lag raha hai, paise mat bhejo"],
        ["free iphone jeetny k liye link click karein"],
    ],
)
demo.launch(share=True)